# **RAG: Does an LLM answer better with passage retrieval?**

This project compares four passage retrieval methods (BM25, dense retrieval, hybrid, re-ranking) by measuring their effect on an LLM's answers to 300 factual questions, with statistical tests to separate real differences from chance.

---
# **Part 1: Retrieving the best passages**

1. The data: Wikipedia passages and questions with their answers
2. Cleaning text and scoring an answer
3. Four ways to retrieve passages: BM25, dense retrieval, hybrid, re-ranking

## 0. Installation and imports

In [1]:
!pip install -q sentence-transformers datasets

In [2]:
import re
import string
import time
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


## 1. The data

[`rag-mini-wikipedia`](https://huggingface.co/datasets/rag-datasets/rag-mini-wikipedia) contains:
- a **corpus** of 3,200 Wikipedia passages, which we will search
- 918 **questions**, each with its expected answer.

In [46]:
corpus = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus", split="passages")
qa = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer", split="test")

passages = [p.strip() for p in corpus["passage"] if p.strip()]
questions = qa["question"]
# One list per question, since a question can have several correct answers
answers = [[str(a)] for a in qa["answer"]]

print(len(passages), "passages |", len(questions), "questions")
pd.DataFrame({"question": questions[:5], "answer": qa["answer"][:5]})

3200 passages | 918 questions


,question,answer
0,Was Abraham Lincoln the sixteenth President of...,yes
1,Did Lincoln sign the National Banking Act of 1...,yes
2,Did his mother die of pneumonia?,no
3,How many long was Lincoln's formal education?,18 months
4,When did Lincoln begin his political career?,1832


## 2. Cleaning text and scoring an answer

- `tokenize` splits a text into words and removes stop words (*the, of, what…*). It is used by BM25.
- `normalize_answer` puts an answer into a standard form so it can be compared (SQuAD normalization).
- `exact_match`, `token_f1` and `contains_answer` score an answer from 0 to 1.

In [5]:
STOPWORDS = set('''
a an and are as at be been but by can did do does for from had has have he her his how i if in into
is it its of on or she so that the their them then there these they this to was we were what when
where which who whom why will with would you your
'''.split())


def strip_accents(text):
    return "".join(c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c))


def tokenize(text):
    words = re.findall(r"\w+", strip_accents(text.lower()))
    return [w for w in words if w not in STOPWORDS]


def normalize_answer(text):
    text = strip_accents(str(text).lower())                                   # lowercase, no accents
    text = "".join(" " if ch in string.punctuation else ch for ch in text)    # punctuation -> space
    text = re.sub(r"\b(a|an|the)\b", " ", text)                               # remove a / an / the
    return " ".join(text.split())                                             # collapse multiple spaces

In [6]:
def exact_match(prediction, golds):
    # 1.0 if the prediction equals at least one of the correct answers
    pred = normalize_answer(prediction)
    return float(any(pred == normalize_answer(g) for g in golds))


def token_f1(prediction, golds):
    # Word-level F1: partial credit when some of the words are right
    pred_words = normalize_answer(prediction).split()
    best = 0.0
    for gold in golds:
        gold_words = normalize_answer(gold).split()
        common = sum((Counter(pred_words) & Counter(gold_words)).values())
        if common == 0:
            continue
        precision = common / len(pred_words)
        recall = common / len(gold_words)
        best = max(best, 2 * precision * recall / (precision + recall))
    return best


def contains_answer(text, golds):
    # 1.0 if a correct answer appears as a whole word in the text
    haystack = f" {normalize_answer(text)} "
    return float(any(f" {normalize_answer(g)} " in haystack for g in golds if normalize_answer(g)))


def is_yes_no(golds):
    return all(normalize_answer(g) in {"yes", "no"} for g in golds)


# Sanity check
print(exact_match("The Impalas.", ["the impalas"]))                    # 1.0
print(token_f1("Michael Moriarty played him", ["Michael Moriarty"]))   # 2* (0.5*1)/(0.5+1) = 0.6667
print(contains_answer("the bus is late", ["us"]))                      # 0.0

1.0
0.6666666666666666
0.0


## 3. The four retrieval methods

They all have the same signature: `search_xxx(question, k)` returns the list of **indices** of the top k passages, from most to least relevant.

We try them on an example question.

In [10]:
def top_k(scores, k):
    # indices of the k highest scores, from highest to lowest
    return list(np.argsort(-scores)[:k])


def show(ids, n_chars=140):
    for rank, i in enumerate(ids, start=1):
        print(f"  {rank}. [{i}] {passages[i][:n_chars]}...")


EXAMPLE = 3
print("Question :", questions[EXAMPLE])
print("Answer   :", answers[EXAMPLE][0])

Question : How many long was Lincoln's formal education?
Answer   : 18 months


### 3.1 BM25: keyword search

A passage's score = sum, over the words of the question, of **IDF × saturated term frequency**:

$$\text{score}(q, d) = \sum_{\text{word} \in q} \text{IDF}(\text{word}) \cdot \frac{tf \cdot (k_1 + 1)}{tf + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{average length}}\right)}$$

- **IDF**: a rare word weighs more.
- **tf and k₁**: each repetition of a word adds a little less than the previous one.
- **b**: long passages are penalized.


**Indexing (once, before any question).** We build a 3,200 × 20,255 matrix: one row per passage, one column per word of the vocabulary (the set of *distinct* words found in the passages).
The (passage, word) cell holds the BM25 weight of that word in that passage, and 0 if the passage does not contain the word.
Since a passage only uses about a hundred words, this matrix is almost entirely zeros. So we do not store it in full: for each word, we keep only the list of passages that contain it, with the corresponding weight.



 **Search (for each question).** We split the question into words. For each passage, we add up the weights of the question words it contains: that is its BM25 score. We then sort the passages by decreasing score and keep the top k (the `top_k` function).

In [14]:
K1, B = 1.5, 0.75

docs = [tokenize(p) for p in passages]                 # each passage split into words
n_docs = len(docs)
doc_len = np.array([len(d) for d in docs])
avg_len = doc_len.mean()

# df: how many passages does each word appear in?
df = Counter(word for d in docs for word in set(d))
# IDF: the rarer a word, the more it weighs
idf = {word: np.log(1 + (n_docs - n + 0.5) / (n + 0.5)) for word, n in df.items()}

# Inverted index: word -> list of (passage index, BM25 weight of the word in that passage)
bm25_index = defaultdict(list)
for i, d in enumerate(docs):
    for word, tf in Counter(d).items():
        length_norm = 1 - B + B * doc_len[i] / avg_len
        weight = idf[word] * tf * (K1 + 1) / (tf + K1 * length_norm)
        bm25_index[word].append((i, weight))

print(f"Vocabulary: {len(bm25_index):,} distinct words")

print("Question :", questions[EXAMPLE])    # reminder of the question

def search_bm25(question, k=5):
    scores = np.zeros(n_docs)
    for word in tokenize(question):
        for i, weight in bm25_index.get(word, []):    # only the passages that contain the word
            scores[i] += weight
    return top_k(scores, k)


show(search_bm25(questions[EXAMPLE])) # passages most likely to contain the correct answer

Vocabulary: 20,255 distinct words
Question : How many long was Lincoln's formal education?
  1. [376] There were few people who strongly or directly influenced Lincolnâs moral and intellectual development and perspectives. There was no teac...
  2. [287] Lincoln's formal education consisted of about 18 months of schooling. Largely self-educated, he read every book he could get his hands on, o...
  3. [385] Lincoln has been memorialized in many city names, notably the capital of Nebraska. Lincoln, Illinois, is the only city to be named for Abrah...
  4. [355] Lincoln's powerful rhetoric defined the issues of the war for the nation, the world, and posterity. His extraordinary command of the English...
  5. [387] Abraham Lincoln's birthday, February 12, was formerly a national holiday, now commemorated as Presidents' Day. However, it is still observed...


### 3.2 Dense retrieval: searching by meaning

Each text becomes a vector of 384 numbers. Texts with similar meanings give nearby vectors.

- `normalize_embeddings=True` scales each vector to length 1. The dot product of two vectors is then their **cosine**.
- We encode the 3,200 passages **only once**. For each question, we just encode the question and compute a matrix-vector product.

In [16]:
encoder = SentenceTransformer("sentence-transformers/multi-qa-MiniLM-L6-cos-v1", device=DEVICE)

passage_vectors = encoder.encode(passages, normalize_embeddings=True, show_progress_bar=True)
print("Passage matrix shape:", passage_vectors.shape)   # (3200, 384)


def search_dense(question, k=5):
    q = encoder.encode(question, normalize_embeddings=True)   # vector of 384 numbers
    scores = passage_vectors @ q                               # 3,200 cosines at once
    return top_k(scores, k)


show(search_dense(questions[EXAMPLE]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/100 [00:00<?, ?it/s]

Passage matrix shape: (3200, 384)
  1. [287] Lincoln's formal education consisted of about 18 months of schooling. Largely self-educated, he read every book he could get his hands on, o...
  2. [292] In 1834, he won election to the state legislature, and after coming across the Commentaries on the Laws of England, began to teach himself l...
  3. [285] Lincoln was just seven years old when, in 1816, the family was forced to make a new start in Perry County (now in Spencer County), Indiana. ...
  4. [288] Young Abraham Lincoln...
  5. [367] An army surgeon, Doctor Charles Leale, initially assessed Lincoln's wound as mortal. The President was taken across the street from the thea...


### 3.3 Hybrid: combining the rankings (Reciprocal Rank Fusion)

BM25 scores and cosines are not on the same scale, so we combine the **ranks**. Each method gives `1 / (60 + rank)` points to every passage in its ranking, and we add them up.

In [19]:
def search_hybrid(question, k=20, depth=100, c=60):       # we compare the top 100 of each method
    points = defaultdict(float)
    for ranking in (search_bm25(question, depth), search_dense(question, depth)):
        for rank, i in enumerate(ranking, start=1):
            points[i] += 1 / (c + rank)
    best = sorted(points, key=points.get, reverse=True)       # from most to fewest points
    return best[:k]


show(search_hybrid(questions[EXAMPLE]))

  1. [287] Lincoln's formal education consisted of about 18 months of schooling. Largely self-educated, he read every book he could get his hands on, o...
  2. [376] There were few people who strongly or directly influenced Lincolnâs moral and intellectual development and perspectives. There was no teac...
  3. [294] Lincoln wrote a series of anonymous letters, published in 1842 in the Sangamon Journal, mocking State Auditor and prominent Democrat James S...
  4. [390] * American School, Lincoln's economic views....
  5. [306] Lincoln was involved in more than 5,100 cases in Illinois alone during his 23-year legal career. Though many of these cases involved little ...
  6. [367] An army surgeon, Doctor Charles Leale, initially assessed Lincoln's wound as mortal. The President was taken across the street from the thea...
  7. [297] The other Lincoln children were born in Springfield, Illinois, and died either during childhood or their teen years. Edward Baker Lincoln wa...
  8. [1909]

### 3.4 Re-ranking: re-reading the top 20 candidates

1. The hybrid method selects the **top 20 candidates** out of the 3,200 passages.
2. The cross-encoder reads each (question, passage) pair and gives it a score.
3. We sort by score and keep the top k.

In [21]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)


def search_rerank(question, k=5, n_candidates=20):
    candidates = search_hybrid(question, n_candidates)
    scores = reranker.predict([(question, passages[i]) for i in candidates])
    order = np.argsort(-scores)                                # from best to worst score
    return [candidates[j] for j in order[:k]]


# The 4 retrieval methods, used in sections 4.3 and 5
METHODS = {
    "BM25": search_bm25,
    "Dense": search_dense,
    "Hybrid": search_hybrid,
    "Hybrid + rerank": search_rerank,
}


show(search_rerank(questions[EXAMPLE]))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  1. [287] Lincoln's formal education consisted of about 18 months of schooling. Largely self-educated, he read every book he could get his hands on, o...
  2. [376] There were few people who strongly or directly influenced Lincolnâs moral and intellectual development and perspectives. There was no teac...
  3. [306] Lincoln was involved in more than 5,100 cases in Illinois alone during his 23-year legal career. Though many of these cases involved little ...
  4. [285] Lincoln was just seven years old when, in 1816, the family was forced to make a new start in Perry County (now in Spencer County), Indiana. ...
  5. [297] The other Lincoln children were born in Springfield, Illinois, and died either during childhood or their teen years. Edward Baker Lincoln wa...


---
# **Part 2: Getting the LLM to answer**

## 4. The LLM and its prompt

We use **Qwen2.5-1.5B-Instruct**

- The **tokenizer** turns text into token IDs, and back.
- The **model** predicts the next token.

In [24]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,   # float16: half the memory on GPU
).to(DEVICE)
model.eval()
print("LLM loaded:", LLM_NAME)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded: Qwen/Qwen2.5-1.5B-Instruct


### 4.1 The prompt

Two messages:
- **the system message** gives the general instructions. We require a **short** answer, otherwise Exact Match would penalize correct but detailed answers;
- **the user message** contains the numbered passages, then the question.

Without passages, this is the **closed-book** version: the model answers from memory.

In [25]:
SYSTEM_PROMPT = (
    "You are a question answering assistant. Reply with the shortest possible answer: "
    "a name, a date, a number or a short phrase (usually 1 to 5 words), never a full sentence. "
    "For yes/no questions reply only 'yes' or 'no'."
)


def build_prompt(question, passage_ids=None):
    if not passage_ids:                                    # closed-book: the question alone
        return f"Question: {question}\nShort answer:"
    numbered = "\n\n".join(f"[{n}] {passages[i]}" for n, i in enumerate(passage_ids, start=1))
    return (
        "Answer the question using the passages below.\n\n"
        f"Passages:\n{numbered}\n\n"
        f"Question: {question}\nShort answer:"
    )


# See exactly what the LLM receives (with 2 passages so it stays readable)
print(build_prompt(questions[EXAMPLE], search_hybrid(questions[EXAMPLE], 2)))

Answer the question using the passages below.

Passages:
[1] Lincoln's formal education consisted of about 18 months of schooling. Largely self-educated, he read every book he could get his hands on, once walking.  just to borrow one  While his favorite book was The Life of George Washington, Lincoln mastered the Bible, Shakespeare, and English and American history, and developed a plain writing style that puzzled audiences more used to grandiose rhetoric. He was also a talented local wrestler and skilled with an ax; some rails he had allegedly split in his youth were exhibited at the 1860 Republican National Convention, as the party celebrated the poor-boy-made-good theme. Lincoln avoided hunting and fishing because he did not like killing animals, even for food. Though he was unusually tall at  , 4 inches and strong, Lincoln spent so much time reading that some neighbors suspected he must be doing it to avoid strenuous manual labor.

[2] There were few people who strongly or directly

### 4.2 The `answer` function

1. We build the two messages, then put them in the format Qwen expects (`apply_chat_template` adds its special tags).
2. We turn the text into tokens.
3. `generate` produces at most 24 new tokens. `do_sample=False`: we **always pick the most likely token**, so the results are reproducible.
4. `generate` returns *prompt + answer*: we keep only the new tokens.
5. We clean up: keep the first line, remove "Answer:", quotes and the final period.

In [39]:
def postprocess(text):
    # Cleans up the answer given by the LLM
    text = text.strip().split("\n")[0]
    text = re.sub(r"^(short answer|answer)\s*:\s*", "", text, flags=re.IGNORECASE)
    return text.strip().strip("\"'*").rstrip(".").strip()


def answer(question, passage_ids=None):
    # The two messages: the instructions, then the passages and the question
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(question, passage_ids)},
    ]
    # Assemble the messages into a single text in Qwen's chat format (special tags)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Turn this text into token IDs and send them to the GPU
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    # Generate the answer token by token (24 at most), always picking the most likely token (do_sample=False) so results are reproducible
    output = model.generate(**inputs, max_new_tokens=24, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
    # generate() returns prompt + answer: keep only the new tokens
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    # Token IDs -> text, then clean up
    return postprocess(tokenizer.decode(new_tokens, skip_special_tokens=True))


### 4.3 Trying the 5 methods on one question

In [48]:
EXAMPLE = 505
q = questions[EXAMPLE]
print("Question         :", q)
print("Expected answer  :", answers[EXAMPLE][0])
print()
print(f"{'Closed-book':<18}:", answer(q))                        # no passage: from memory
for name, search in METHODS.items():
    print(f"{name:<18}:", answer(q, search(q, 5)))               # with the method's top 5 passages

Question         : What centred in Sierra?
Expected answer  : The leopard men

Closed-book       : No
BM25              : West African secret society called Leopard men
Dense             : Ghana
Hybrid            : Cerro Catedral
Hybrid + rerank   : West African secret society called Leopard men


## 5. Which method is best?

Experiment: we use **the same questions, the same LLM, the same prompt**. Only the context changes:
- *Closed-book*: no passage
- then the top 5 passages from each of the 4 retrieval methods.

For each answer, we compute Exact Match (EM), F1 and `contains` (does the correct answer appear in the prediction?). We then average per method.

We also keep `context_has_answer`: was the correct answer in the passages provided? This will be used for error analysis.

We draw **300 questions at random**, with a fixed seed so the experiment can be reproduced exactly.

In [52]:
N_QA = 300
rng = np.random.default_rng(42)                                      # fixed seed: always the same questions
qa_ids = sorted(int(i) for i in rng.choice(len(questions), size=N_QA, replace=False))

SYSTEMS = {"Closed-book": None, **METHODS}                           # None = no retrieval
rows = []
for name, search in SYSTEMS.items():
    start = time.time()
    for i in qa_ids:
        q, golds = questions[i], answers[i]
        ids = search(q, 5) if search else []
        pred = answer(q, ids)
        rows.append({
            "method": name,
            "question": q,
            "expected answer": golds[0],
            "prediction": pred,
            "em": exact_match(pred, golds),
            "f1": token_f1(pred, golds),
            "contains": contains_answer(pred, golds),
            # was the correct answer in the passages? (meaningless for closed-book and for yes/no)
            "context_has_answer": (float(any(contains_answer(passages[j], golds) for j in ids))
                                   if ids and not is_yes_no(golds) else np.nan),
        })
    print(f"{name:<18} done in {time.time() - start:.0f} s")

results = pd.DataFrame(rows)
results.to_csv("qa_results.csv", index=False)     # save, in case Colab disconnects

Closed-book        done in 51 s
BM25               done in 92 s
Dense              done in 114 s
Hybrid             done in 120 s
Hybrid + rerank    done in 119 s


In [53]:
summary = results.groupby("method")[["em", "f1", "contains"]].mean().loc[list(SYSTEMS)]
summary.round(3)

,em,f1,contains
method,,,
Closed-book,0.320,0.353,0.323
BM25,0.553,0.643,0.570
Dense,0.540,0.617,0.563
Hybrid,0.577,0.656,0.603
Hybrid + rerank,0.583,0.670,0.607


### 5.1 The answers side by side

For each question, the expected answer and the prediction of each version.

In [55]:
side_by_side = results.pivot_table(index=["question", "expected answer"], columns="method",
                                   values="prediction", aggfunc="first")[list(SYSTEMS)]
side_by_side.head(15)

,method,Closed-book,BM25,Dense,Hybrid,Hybrid + rerank
question,expected answer,,,,,
Adams ' opponents were what?,Democratic Republicans,Democrats,Democratic-Republicans,Thomas Jefferson and Aaron Burr,Democratic-Republican,Democratic-Republicans
Are Gray Wolves native to North America?,Yes,Yes,Yes,Yes,Yes,Yes
Are Kangaroos Shy?,Yes,No,Yes,Yes,Yes,Yes
Are ducks in the Arctic Northern Hemisphere migratory?,Some are,Yes,Yes,Yes,Yes,Yes
Are large pythons potential prey for leopards?,yes,Yes,Yes,Yes,Yes,Yes
Are leopards circular?,No,No,No,No,No,No
Are nationals referred to by the plural : Liechtensteiners ?,Yes,No,Yes,Yes,Yes,Yes
Are otters playful animals?,yes,Yes,Yes,Yes,Yes,Yes
Are otters totem animals?,Yes,Yes,Yes,Yes,Yes,Yes


### 5.2 Conclusion (300 questions, k = 5 passages)

| Method | EM | F1 | contains |
|---|---|---|---|
| Closed-book | 0.320 | 0.353 | 0.323 |
| BM25 | 0.553 | 0.643 | 0.570 |
| Dense | 0.540 | 0.617 | 0.563 |
| Hybrid | 0.577 | 0.656 | 0.603 |
| Hybrid + rerank | 0.583 | 0.670 | 0.607 |

**1. Passage retrieval clearly improves the LLM's answers.**
Without passages, the model gives an exactly correct answer to 32% of the questions. With the 5 passages from any method, it exceeds 54%, and reaches 58% with the hybrid methods.
With 300 questions, the uncertainty on each score is about ±5.6 points (95% confidence interval). A gap of this size therefore cannot be explained by chance.

**2. The hybrid methods appear to perform best, but the differences between retrieval methods remain small.**
Hybrid (0.577) and hybrid with re-ranking (0.583) are ahead of BM25 (0.553) and dense retrieval (0.540).
However, these differences are too small to confirm that they are not due to the particular sample. (They will need to be confirmed with a paired test, in section 6.)


**3. The model's errors are about content, not format.**
The gap between `contains` and EM is only 2 to 3 points for every method: when the model finds the correct answer, it almost always gives it in the expected format.
The gap between F1 and EM, about 8 points, comes from partially correct answers, such as "West African secret society called Leopard men" for "The leopard men".



## 6. Are the differences between methods significant?

The scores in section 5 are measured on **a single sample** of 300 questions. With 300 other questions, we would have obtained slightly different numbers. So we need to check that the differences between methods are not due to the particular sample.

**The bootstrap** estimates this uncertainty from our single sample:
1. We draw 300 questions at random from our 300, **with replacement**.
2. We compute the mean score on this new sample.
3. We repeat this 2,000 times.
4. We compute the **95% confidence interval**.

**The paired bootstrap** is used to compare two methods A and B. Since they answered the **same** questions, we compute for each question the score difference A − B: +1 if only A is right, −1 if only B is right, 0 if they agree. We then apply the bootstrap to the mean of these differences. If the confidence interval **does not contain 0**, the difference is **significant**.

Comparing question by question is more precise than comparing two separate intervals: the difficulty specific to each question cancels out, and only the questions where the two methods disagree count.

In [60]:
def bootstrap_ci(values, n_boot=2000, seed=0):
    # 95% confidence interval of the mean, by bootstrap
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    # 2,000 samples of the same size, drawn with replacement (each row = one sample of indices)
    idx = rng.integers(0, len(values), size=(n_boot, len(values)))
    means = values[idx].mean(axis=1)                       # the mean of each sample
    low, high = np.percentile(means, [2.5, 97.5])
    return values.mean(), low, high


# Scores of each method, in the same question order (qa_ids) for every method
scores = {name: {m: results.loc[results["method"] == name, m].to_numpy() for m in ["em", "f1"]}
          for name in SYSTEMS}

### 6.1 Confidence interval of each method

In [61]:
ci_rows = []
for name in SYSTEMS:
    row = {"method": name}
    for metric in ["em", "f1"]:
        mean, low, high = bootstrap_ci(scores[name][metric])
        row[metric] = round(mean, 3)
        row[f"95% CI {metric}"] = f"[{low:.3f}, {high:.3f}]"
    ci_rows.append(row)

pd.DataFrame(ci_rows).set_index("method")

,em,95% CI em,f1,95% CI f1
method,,,,
Closed-book,0.320,"[0.263, 0.370]",0.353,"[0.300, 0.402]"
BM25,0.553,"[0.497, 0.607]",0.643,"[0.593, 0.692]"
Dense,0.540,"[0.480, 0.597]",0.617,"[0.565, 0.665]"
Hybrid,0.577,"[0.520, 0.630]",0.656,"[0.609, 0.706]"
Hybrid + rerank,0.583,"[0.527, 0.637]",0.670,"[0.622, 0.716]"


**Results**

The closed-book CI does not overlap any of the CIs of the retrieval methods. Retrieval clearly does better.

Among the four retrieval methods, the CIs overlap widely. Looking at them separately, we cannot tell them apart.

### 6.2 Comparing two methods: the paired bootstrap (on EM)

The separate CIs overlap, but that is not enough to conclude that the methods are equivalent. Since all the methods answered the same 300 questions, we can compare them question by question.

For two methods A and B, we compute for each question the EM difference, A − B:

| Question | EM of A | EM of B | A − B |
|---|---|---|---|
| 1 | 1 | 1 | 0 (both right) |
| 2 | 0 | 0 | 0 (both wrong) |
| 3 | 1 | 0 | **+1** (only A is right) |
| 4 | 0 | 1 | **−1** (only B is right) |

The mean of these differences is the EM gap between A and B. We then apply the bootstrap to this column of differences: 2,000 draws with replacement, 2,000 means, then the interval between the 2.5th and 97.5th percentiles.

**Decision rule:**
- if the CI of the difference **does not contain 0**, the difference is **significant**: one method is genuinely better;
- if it **contains 0**, we cannot conclude: the true difference could be zero.



In the results table, the **"A only correct"** and **"B only correct"** columns give the number of these disagreements: the whole comparison rests on them.

In [62]:
def compare(a, b, metric="em"):
    sa, sb = scores[a][metric], scores[b][metric]
    mean, low, high = bootstrap_ci(sa - sb)                # question-by-question difference
    return {
        "comparison": f"{a} − {b}",
        "Δ EM": round(mean, 3),
        "95% CI": f"[{low:+.3f}, {high:+.3f}]",
        "significant": "yes" if not (low <= 0 <= high) else "no",
        f"{a} only correct": int(((sa == 1) & (sb == 0)).sum()),   # questions where only method A is right
        f"{b} only correct": int(((sa == 0) & (sb == 1)).sum()),   # questions where only method B is right
    }


comparisons = pd.DataFrame([
    compare("BM25", "Closed-book"),
    compare("Hybrid", "BM25"),
    compare("Hybrid", "Dense"),
    compare("BM25", "Dense"),
    compare("Hybrid + rerank", "Hybrid"),
]).set_index("comparison")
comparisons

,Δ EM,95% CI,significant,BM25 only correct,Closed-book only correct,Hybrid only correct,Dense only correct,Hybrid + rerank only correct
comparison,,,,,,,,
BM25 − Closed-book,0.233,"[+0.180, +0.290]",yes,80.0,10.0,NaN,NaN,NaN
Hybrid − BM25,0.023,"[-0.003, +0.050]",no,5.0,NaN,12.0,NaN,NaN
Hybrid − Dense,0.037,"[+0.013, +0.063]",yes,NaN,NaN,13.0,2.0,NaN
BM25 − Dense,0.013,"[-0.020, +0.047]",no,15.0,NaN,NaN,11.0,NaN
Hybrid + rerank − Hybrid,0.007,"[-0.020, +0.033]",no,NaN,NaN,7.0,NaN,9.0


**Results**

| Comparison (A − B) | Δ EM | 95% CI | Significant | A only correct | B only correct |
|---|---|---|---|---|---|
| BM25 − Closed-book | +0.233 | [+0.180, +0.290] | yes | 80 | 10 |
| Hybrid − BM25 | +0.023 | [−0.003, +0.050] | no | 12 | 5 |
| Hybrid − Dense | +0.037 | [+0.013, +0.063] | yes | 13 | 2 |
| BM25 − Dense | +0.013 | [−0.020, +0.047] | no | 15 | 11 |
| Hybrid + rerank − Hybrid | +0.007 | [−0.020, +0.033] | no | 9 | 7 |

**1. Retrieval very clearly improves the LLM's answers.**
Of the 90 questions where BM25 and closed-book disagree, BM25 gives the correct answer 80 times. That leaves 10 questions where the model knew the answer from memory, but the passages misled it.

**2. Hybrid is significantly better than dense retrieval.**
Taken separately, their CIs overlap. Yet the paired test shows that, on the 15 questions where they disagree, hybrid is right 13 times. Combining keywords (BM25) and meaning (dense retrieval) therefore fixes some of the errors of dense retrieval alone.

**3. Hybrid's advantage over BM25 is not demonstrated.**
Hybrid wins 12 disagreements to 5, but the CI [−0.003, +0.050] just barely contains 0. It is a favorable trend that would need to be confirmed on more questions.

**4. BM25 and dense retrieval are equivalent** on this corpus (15 disagreements won to 11, difference not significant).

**5. Re-ranking brings no gain.**
With 9 disagreements won to 7, it is equivalent to hybrid alone, even though it adds a model and compute time. On this corpus, BM25 + dense fusion is enough.


---
# **Overall conclusion**

## The question
Does an LLM answer factual questions better when given Wikipedia passages (RAG), and which passage retrieval method is the most effective?

## What was done
- **4 retrieval methods** on a corpus of 3,200 passages: BM25 (keywords, implemented by hand), dense retrieval (embeddings), hybrid (fusion of both rankings with Reciprocal Rank Fusion) and hybrid + re-ranking (cross-encoder).
- **The same LLM** (Qwen2.5-1.5B-Instruct) answers **the same 300 questions**, without passages (*closed-book*) and then with the 5 passages from each method.
- **Evaluation** with Exact Match, F1 and `contains`, with bootstrap confidence intervals and paired comparisons.

## Results
1. **Passage retrieval very clearly improves the answers.** EM rises from **0.32** without passages to **0.58** with the hybrid methods (+26 points). This is the most robust result of the study: the gap is highly significant.
2. **Hybrid is the most reliable method.** It significantly beats dense retrieval (13 of 15 disagreements won) and is ahead of BM25, although that advantage is not yet demonstrated (CI [−0.003, +0.050]).
3. **Re-ranking brings no gain** on this corpus, even though it adds compute time.
4. **The remaining errors are about content, not format**: the gap between `contains` and EM is only 2 to 3 points.

**Recommendation:** for this kind of corpus, **hybrid BM25 + dense retrieval, without re-ranking**, offers the best trade-off between answer quality and compute cost.

## Limitations
- **A single, small LLM** (1.5 billion parameters). With a larger model, the closed-book score would be higher and the benefit of retrieval probably smaller.
- **A small corpus** (3,200 passages on related topics). On a corpus of several million passages, retrieval would be harder and the differences between methods probably more pronounced.
- **A single value of k** (5 passages) and untuned parameters (BM25's k₁ and b, the RRF constant, 20 candidates for re-ranking).
- **300 questions**: the differences between retrieval methods (2 to 4 points) are close to the detection threshold. Five tests were run without correcting for multiple comparisons: the only significant difference between retrieval methods (hybrid vs. dense) remains to be confirmed.
- **Exact Match is strict**: a correct answer phrased differently ("18 months" vs. "a year and a half") is counted as wrong.

## Next steps
- **Error analysis**: when the LLM is wrong, was the correct answer in the passages? This would tell whether to improve retrieval or the model. The `context_has_answer` column already makes this possible.
- Study the **10 questions where the passages misled the model**, even though it knew the answer from memory.
- Try **several values of k** and a **larger LLM**.